In [6]:
import pickle
import numpy as np
import gensim
from spec2vec import SpectrumDocument, Spec2Vec


In [ ]:
# Load the cleaned spectra data for pesticides and pharmaceuticals
with open("/Users/timvandriel/Desktop/CSCI558/project/data/pesticides_clean_spectra.pkl", "rb") as f:
    pest_spectra = pickle.load(f)
    
with open("/Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_clean_spectra.pkl", "rb") as f:
    pharma_spectra = pickle.load(f)

In [8]:
# 1. Load weights
model = gensim.models.Word2Vec.load("/Users/timvandriel/Desktop/CSCI558/project/models/spec2vec_positive.model")
print(f"Model loaded. Vocabulary size: {len(model.wv.key_to_index):,} tokens.")

# 2. Tokenize cleaned spectra
pest_docs = [SpectrumDocument(s, n_decimals=2) for s in pest_spectra]
pharma_docs = [SpectrumDocument(s, n_decimals=2) for s in pharma_spectra]

# 3. Check vocabulary coverage on document 0
doc0_words = pest_docs[0].words
known = [w for w in doc0_words if w in model.wv.key_to_index]
print(f"Doc 0 Tokens: {len(doc0_words)} total | {len(known)} in vocabulary ({len(known)/len(doc0_words)*100:.1f}%)")

Model loaded. Vocabulary size: 115,910 tokens.
Doc 0 Tokens: 30 total | 30 in vocabulary (100.0%)


In [11]:
spec2vec_similarity = Spec2Vec(model=model, intensity_weighting_power=0.5, allowed_missing_percentage=15.0)

def compute_similarity_matrix(docs):
    print(f"Computing similarity matrix for {len(docs)} documents...")
    # compute similarity matrix
    sim_matrix = spec2vec_similarity.matrix(docs, docs)
    
    # Cast to float32
    sim_matrix = sim_matrix.astype(np.float32)
    
    # Sanity checks
    assert np.allclose(np.diag(sim_matrix), 1.0, atol=1e-3), "Diagonal self-similarity should be ~1.0"
    assert np.allclose(sim_matrix, sim_matrix.T, atol=1e-3), "Spec2Vec matrix should be symmetric"
    
    # Distribution summary of off-diagonal pairs
    off_diag = sim_matrix[np.triu_indices_from(sim_matrix, k=1)]
    print(f"Off-diagonal Spec2Vec -> Mean: {off_diag.mean():.4f}, Median: {np.median(off_diag):.4f}, Max: {off_diag.max():.4f}")
    
    return sim_matrix
    
    

In [12]:
pest_sim_matrix = compute_similarity_matrix(pest_docs)
pharma_sim_matrix = compute_similarity_matrix(pharma_docs)

# Save the computed spectral similarity matrix
out_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/pest_spec2vec_matrix.npy"
np.save(out_file, pest_sim_matrix)
print(f"Saved: {out_file} (shape: {pest_sim_matrix.shape})")

out_file = f"/Users/timvandriel/Desktop/CSCI558/project/data/pharma_spec2vec_matrix.npy"
np.save(out_file, pharma_sim_matrix)
print(f"Saved: {out_file} (shape: {pharma_sim_matrix.shape})")

Computing similarity matrix for 653 documents...
Off-diagonal Spec2Vec -> Mean: 0.0558, Median: 0.0426, Max: 0.9987
Computing similarity matrix for 377 documents...
Off-diagonal Spec2Vec -> Mean: 0.1485, Median: 0.1434, Max: 0.9625
Saved: /Users/timvandriel/Desktop/CSCI558/project/data/pest_spec2vec_matrix.npy (shape: (653, 653))
Saved: /Users/timvandriel/Desktop/CSCI558/project/data/pharma_spec2vec_matrix.npy (shape: (377, 377))


In [14]:
from scipy.stats import spearmanr, pearsonr
pest_matrix = np.load("/Users/timvandriel/Desktop/CSCI558/project/data/pesticides_tanimoto_matrix.npy")
pharma_matrix = np.load("/Users/timvandriel/Desktop/CSCI558/project/data/pharmaceuticals_tanimoto_matrix.npy")
def evaluate_alignment(ground_truth_mat, pred_mat, name="Pesticides"):
    # Extract upper triangle excluding self-similarity diagonal
    triu_idx = np.triu_indices_from(ground_truth_mat, k=1)
    y_true = ground_truth_mat[triu_idx]
    y_pred = pred_mat[triu_idx]
    
    pearson_corr, _ = pearsonr(y_true, y_pred)
    spearman_corr, _ = spearmanr(y_true, y_pred)
    
    print(f"\n--- {name}: Spec2Vec vs. Structural Tanimoto ---")
    print(f"Total evaluated pairs: {len(y_true):,}")
    print(f"Pearson Correlation (r):  {pearson_corr:.4f}")
    print(f"Spearman Correlation (ρ): {spearman_corr:.4f}")

evaluate_alignment(pest_matrix, pest_sim_matrix, "Pesticides")
evaluate_alignment(pharma_matrix, pharma_sim_matrix, "Pharmaceuticals")


--- Pesticides: Spec2Vec vs. Structural Tanimoto ---
Total evaluated pairs: 212,878
Pearson Correlation (r):  0.4070
Spearman Correlation (ρ): 0.1107

--- Pharmaceuticals: Spec2Vec vs. Structural Tanimoto ---
Total evaluated pairs: 70,876
Pearson Correlation (r):  0.2262
Spearman Correlation (ρ): 0.1776


In [15]:
import numpy as np
import pandas as pd

def stratify_scores(ground_truth_mat, pred_mat, name="Pesticides"):
    triu_idx = np.triu_indices_from(ground_truth_mat, k=1)
    y_true = ground_truth_mat[triu_idx]
    y_pred = pred_mat[triu_idx]
    
    # Define Tanimoto similarity bins
    bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    labels = ["0.0-0.2 (Unrelated)", "0.2-0.4 (Distant)", "0.4-0.6 (Moderate)", "0.6-0.8 (Close)", "0.8-1.0 (Identical/Isomer)"]
    
    bin_assignments = pd.cut(y_true, bins=bins, labels=labels, include_lowest=True)
    df = pd.DataFrame({"true_tanimoto": y_true, "spec2vec": y_pred, "bin": bin_assignments})
    
    summary = df.groupby("bin", observed=False)["spec2vec"].agg(
        Pair_Count="count",
        Mean_Spec2Vec="mean",
        Median_Spec2Vec="median",
        Std_Spec2Vec="std"
    ).reset_index()
    
    print(f"\n{'='*55}")
    print(f"Spec2Vec Scores by Structural Similarity Bin: {name.upper()}")
    print(f"{'='*55}")
    print(summary.to_string(index=False))
    return summary

pest_strat = stratify_scores(pest_matrix, pest_sim_matrix, "Pesticides")
pharma_strat = stratify_scores(pharma_matrix, pharma_sim_matrix, "Pharmaceuticals")


Spec2Vec Scores by Structural Similarity Bin: PESTICIDES
                       bin  Pair_Count  Mean_Spec2Vec  Median_Spec2Vec  Std_Spec2Vec
       0.0-0.2 (Unrelated)      193301       0.047781         0.040937      0.078111
         0.2-0.4 (Distant)       16193       0.091986         0.054272      0.155269
        0.4-0.6 (Moderate)        1636       0.203878         0.108849      0.238561
           0.6-0.8 (Close)         520       0.334611         0.264871      0.257649
0.8-1.0 (Identical/Isomer)        1228       0.529139         0.621329      0.369199

Spec2Vec Scores by Structural Similarity Bin: PHARMACEUTICALS
                       bin  Pair_Count  Mean_Spec2Vec  Median_Spec2Vec  Std_Spec2Vec
       0.0-0.2 (Unrelated)       68744       0.145857         0.141594      0.118253
         0.2-0.4 (Distant)        1889       0.210200         0.196003      0.126039
        0.4-0.6 (Moderate)         130       0.335049         0.322481      0.172331
           0.6-0.8 (Close)   